The following exercises are meant to be solved by gathering the bash commands incrimentally in two scripts, one for ex 1.* the other for ex 2.* 

# EX 1

1\.a Make a new directory called `students` in your home. Download a csv file with the list of students of this lab from [here](https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv) (use the `wget` command) and copy that to `students`. First check whether the file is already there

```bash
#!/bin/bash
```

```bash
cd $HOME             # va a '/home/seccob'
mkdir students       # crea cartella nella home
# download del file:
wget -q https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv    # con -O rinomino, con -q evito si generi la cartella di diagnostica di wget
cp LCP_22-23_students.csv ./students    # lo copio nella nuova cartella
cd students

# check if the file is there
if [ ! -f "./LCP_22-23_students.csv" ] 
then echo "the file downloaded does not exist!"
exit 1
fi
```

NOTA mia: occhio a copiare esattamente il link dalla cella della consegna, e non quello visualizzato sulla barra del browser una volta aperto il link, se no lo script dà problemi

1\.b Make two new files, one containing the students belonging to PoD, the other to Physics.

```bash
touch pod.txt   # creo i file vuoti
touch phy.txt
grep -c "PoD" LCP_22-23_students.csv                  # I count the rows with text "PoD" (they are 67)
grep -c "Physics" LCP_22-23_students.csv              # I count the rows with text "PoD" (they are 5)
grep "PoD" LCP_22-23_students.csv > pod.txt           # scrivo per intero le righe con "PoD", redirigendo l'output su file
grep "Physics" LCP_22-23_students.csv > phy.txt       # scrivo per intero le righe con "PoD", redirigendo l'output su file
```

1\.c For each letter of the alphabet, count the number of students whose surname starts with that letter. 

```bash
for i in {A..Z}; do grep -c "^$i" LCP_22-23_students.csv; done
```

1\.d Find out which is the letter with most counts.

```bash
best=""
tmp=0
for letter in {A..Z}; 
do
    count=$(grep -c "^$letter" LCP_22-23_students.csv)
    if [ $count -gt $tmp ]
    then 
        tmp=$count
        best=$letter
    fi
done
echo "The letter with most counts is $best with $tmp occurrences"
```

1\.e Assume an obvious numbering of the students in the file (first line is 1, second line is 2, etc.), group students "modulo 18", i.e. 1,19,37,.. 2,20,38,.. etc. and put each group in a separate file  

```bash
for rest in {1..18}; do touch "${rest}.txt"; done    # we create all files
tot=$(wc -l < LCP_22-23_students.csv)                # total lines of our file
for (( i=2; i<=$tot; i++ )); do                      # we start in 2 to exclude the first raw with names of columns
  group=$(( (i-1) %18 ))
  sed -n "${i}p" LCP_22-23_students.csv >> "$group.txt"
done
```

### WHOLE EXERCISE 1 SCRIPT (OPTIMIZED):
```bash
#!/bin/bash
# cd $HOME             # va a '/home/seccob'
mkdir students       # crea cartella nella home
# download del file:
wget -q https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv    # con -O rinomino, con -q evito si generi la cartella di diagnostica di wget
tail -n +2 LCP_22-23_students.csv > LCP_22-23.csv      # I copy the file without first row (that has column names)
cp LCP_22-23.csv ./students                            # lo copio nella nuova cartella
cd students

# check if the file is there
if [ ! -f "./LCP_22-23.csv" ] 
then 
    echo "the file downloaded does not exist!"
    exit 1
else
    echo "All good till now"
fi

touch pod.txt   # creo i file vuoti
touch phy.txt
grep -c "PoD" LCP_22-23.csv                  # I count the rows with text "PoD" (they are 67)
grep -c "Physics" LCP_22-23.csv              # I count the rows with text "PoD" (they are 5)
grep "PoD" LCP_22-23.csv > pod.txt           # scrivo per intero le righe con "PoD", redirigendo l'output su file
grep "Physics" LCP_22-23.csv > phy.txt       # scrivo per intero le righe con "PoD", redirigendo l'output su file

for i in {A..Z}; do 
    count=$(grep -c "^$i" LCP_22-23.csv)
    echo "Letter $i has $count occurrences"
done

best=""
tmp=0
for letter in {A..Z}; 
do
    count=$(grep -c "^$letter" LCP_22-23.csv)
    if [ $count -gt $tmp ]
    then 
        tmp=$count
        best=$letter
    fi
done
echo "The letter with most counts is $best with $tmp occurrences"

for rest in {1..18}; do touch "${rest}.txt"; done    # we create all files
tot=$(wc -l < LCP_22-23.csv)                # total lines of our file
for (( i=1; i<=$tot; i++ )); do                   
  group=$(( (i-1) %18 +1 ))
  sed -n "${i}p" LCP_22-23.csv >> "$group.txt"
done


```

# EX 2

2.a Make a copy of the file `data.csv` removing the metadata and the commas between numbers; call it `data.txt`

```bash
#!/bin/bash
```

```bash
#Instruction sed 's/a/b/g' substitues character 'a' with character 'b' in every occurrence of a row (the final 'g' means 'global')
grep -v "^#" data.csv | sed "s/,//g" > data.txt
```

2\.b How many even numbers are there?

```bash
# L'opzione -o fa in modo che grep stampi solo le parti del testo che corrispondono alla regex (e non l'intera riga). 
# \b è un metacarattere che indica un confine di parola. Serve per assicurarsi che stiamo catturando numeri "isolati" e non parti di altri numeri.
# [0-9] è una classe di caratteri che rappresenta tutte le cifre decimali da 0 a 9
# L'asterisco * dopo [0-9] significa "zero o più occorrenze"
# con wc - l poi conto tutte le righe di output (che sono una per ogni numero pari trovato)
grep -o '\b[0-9]*[02468]\b' data.txt | wc -l
```

2\.c Distinguish the entries on the basis of `sqrt(X^2 + Y^2 + Z^2)` is greater or smaller than `100*sqrt(3)/2`. Count the entries of each of the two groups 

```bash
threshold=$(echo "scale=3; 100 * sqrt(3) / 2" | bc -l)          # limit value
count_greater=0
count_smaller=0
# facciamo un ciclo che legga ciascun elemento di ogni riga
while read -r row; do
    set -- $row         # now my positional variables $1,$2.. are the elements of the row
    xa=$1
    ya=$2
    za=$3
    xb=$4
    yb=$5
    zb=$6
    da=$(echo "scale=3; sqrt($xa^2 + $ya^2 + $za^2)" | bc -l)
    db=$(echo "scale=3; sqrt($xb^2 + $yb^2 + $zb^2)" | bc -l)
    if (( $(echo "$da > $threshold" | bc -l) )); then
        (( count_greater++ ))
    else (( count_smaller++ ))
    fi
    if (( $(echo "$db > $threshold" | bc -l) )); then
        (( count_greater++ ))
    else (( count_smaller++ ))
    fi
done < data.txt
echo "Numbers greater than threshold: $count_greater"
echo "Numbers smaller than threshold: $count_smaller"
```

#### alternative 2.c:
```bash
for line in $(cat data.txt); do    # tutto il file di dati diventa un'unica lunga riga di elementi
    for n in $line; do             # qua ciclo ciascun elemento della giga riga
    ....
```

2\.d Make `n` copies of data.txt (with `n` an input parameter of the script), where the i-th copy has all the numbers divided by i (with `1<=i<=n`).

```bash
echo "Write number (n>1) of copies of file.txt (modified): "
read num_copie            # numero di volte di cui fare una copia, immesso da shell
for ((i=1; i<=num_copie; i++)); do
    while read -r row; do
        for x in $row; do
            result=$(echo "scale=6; $x / $i" | bc -l)
            echo -n "$result " >> "data${i}.txt"          # stampa il valore di result senza aggiungere una nuova riga alla fine
        done
        echo "" >> "data${i}.txt"                         # stampa un "a capo"
    done < data.txt 
done
```